# 06 — Bronze: Compras.gov (Contratos)

## Purpose
Ingests raw contract data from the Compras.gov landing zone into the Bronze layer.
Reads JSON files from `data/raw/compras_gov/`, applies minimal transformations
(technical columns only), and writes partitioned Parquet files to `data/bronze/compras_contratos/`.

## Input
- `data/raw/compras_gov/{YYYY-MM}.json` — one JSON file per month (flat structure)
- 64 months available (2021-01 → 2026-04) — ~761k records total

## Output
- `data/bronze/compras_contratos/_year_month={YYYY-MM}/data.parquet` — partitioned by month
- Schema: 38 source columns (all STRING) + `_rescued_data` + 5 technical columns
- `db/schema_drift_log.json` — drift events (non-null _rescued_data)

## Steps
1. Imports and configuration
2. Schema definition
3. Process months
4. Validation
5. Ops registration

## Notes
- All source columns cast to STRING — no business rules, no type casting
- `_rescued_data`: any column in source JSON not in BRONZE_COLUMNS is captured
  as a JSON blob per record — pipeline never stops, drift is logged
- `unidadesRequisitantes` serialized to JSON string (was None or str in source)
- `contratoExcluido` cast to STRING — Silver applies the filter
- Idempotent — existing partitions are overwritten on re-run
- Local engine: DuckDB reading JSON → writing Parquet
- Raw file structure: `{YYYY-MM}.json` flat (no subdirectory) — matches 04_bootstrap_compras output
- ADRs: ADR-002 (STRING for CNPJ), ADR-006 (Compras.gov Módulo 09)


## Step 1 — Imports and configuration

In [ ]:
import json
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import get_project_root, to_sql_path
from duckdb_utils import get_connection
from validation import CheckSuite
from bootstrap_log import append_log, make_entry
from pipeline import check_landing_gate

import duckdb

# ---------------------------------------------------------------------------
# Project root and paths
# ---------------------------------------------------------------------------
PROJECT_ROOT = get_project_root()

RAW_ROOT    = PROJECT_ROOT / "data" / "raw"    / "compras_gov"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze" / "compras_contratos"
LOG_PATH    = PROJECT_ROOT / "db"  / "bootstrap_log.json"

# ---------------------------------------------------------------------------
# Landing gate check — do NOT proceed if landing zone is not validated
# ---------------------------------------------------------------------------
check_landing_gate(PROJECT_ROOT)

# ---------------------------------------------------------------------------
# Pipeline metadata
# ---------------------------------------------------------------------------
SOURCE_ID  = "compras_gov"
BATCH_ID   = str(uuid.uuid4())
STARTED_AT = datetime.now(timezone.utc).isoformat()

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"RAW_ROOT     : {RAW_ROOT}")
print(f"BRONZE_ROOT  : {BRONZE_ROOT}")
print(f"BATCH_ID     : {BATCH_ID}")
print(f"STARTED_AT   : {STARTED_AT}")


## Step 2 — Schema definition

In [ ]:
# ---------------------------------------------------------------------------
# Bronze schema — 38 source columns + _rescued_data + 5 technical columns
# All source columns as STRING — no type casting at Bronze layer
# Confirmed against EDA: 38 columns stable across 64 months (2021-01 → 2026-04)
# _rescued_data: local equivalent of Databricks rescuedDataColumn option
# ---------------------------------------------------------------------------

BRONZE_COLUMNS = [
    # Org / unit identifiers
    "codigoOrgao",
    "nomeOrgao",
    "codigoUnidadeGestora",
    "nomeUnidadeGestora",
    "codigoUnidadeGestoraOrigemContrato",
    "nomeUnidadeGestoraOrigemContrato",
    "codigoUnidadeRealizadoraCompra",
    "nomeUnidadeRealizadoraCompra",

    # Contract identifiers
    "numeroContrato",
    "numeroCompra",
    "numeroControlePncpContrato",
    "numeroControlePncpCompra",
    "idCompra",
    "processo",

    # Classification
    "receitaDespesa",
    "codigoModalidadeCompra",
    "nomeModalidadeCompra",
    "codigoTipo",
    "nomeTipo",
    "codigoCategoria",
    "nomeCategoria",
    "codigoSubcategoria",
    "nomeSubcategoria",

    # Supplier
    "niFornecedor",
    "nomeRazaoSocialFornecedor",

    # Contract content
    "objeto",
    "informacoesComplementares",

    # Dates
    "dataVigenciaInicial",
    "dataVigenciaFinal",
    "dataHoraInclusao",
    "dataHoraExclusao",

    # Values
    "valorGlobal",
    "valorParcela",
    "valorAcumulado",
    "totalDespesasAcessorias",
    "numeroParcelas",

    # Flags
    "contratoExcluido",

    # Nested — serialized to STRING via to_json()
    "unidadesRequisitantes",
]

# Technical columns added by the pipeline
TECHNICAL_COLUMNS = [
    "_rescued_data",   # JSON blob of unexpected source columns (None = no drift)
    "_source_id",
    "_batch_id",
    "_ingested_at",
    "_source_file",
    "_year_month",
]

ALL_COLUMNS = BRONZE_COLUMNS + TECHNICAL_COLUMNS

print(f"Source columns   : {len(BRONZE_COLUMNS)}")
print(f"Technical columns: {len(TECHNICAL_COLUMNS)}")
print(f"Total columns    : {len(ALL_COLUMNS)}")
print()
for col in BRONZE_COLUMNS:
    print(f"  {col}")


## Step 3 — Process months

In [ ]:
DRIFT_LOG_PATH = PROJECT_ROOT / "db" / "schema_drift_log.json"


def _get_source_columns(source_path: str, con) -> set:
    """
    Return the set of column names present in the source JSON file.
    DuckDB infers the schema from the actual data.
    """
    result = con.execute(
        f"DESCRIBE SELECT * FROM read_json_auto('{source_path}', "
        f"maximum_object_size=33554432) LIMIT 0"
    ).fetchall()
    return {row[0] for row in result}


def _inferred_as_json(source_path: str, con) -> set:
    """
    Return columns inferred as STRUCT or LIST by DuckDB.
    Used to decide whether to apply to_json() before VARCHAR cast.

    Notes
    -----
    `unidadesRequisitantes` is None in 71.6% of records but inferred as STRUCT
    when the field is present. to_json() must be applied to avoid type errors.
    """
    try:
        result = con.execute(
            f"DESCRIBE SELECT * FROM read_json_auto('{source_path}', "
            f"maximum_object_size=33554432) LIMIT 0"
        ).fetchall()
        return {
            row[0] for row in result
            if "STRUCT" in row[1].upper() or "LIST" in row[1].upper()
        }
    except Exception:
        return set()


def _build_rescued_data(record: dict, known_columns: set) -> str | None:
    """
    Build the _rescued_data JSON blob for a single record.

    Captures any key in the record that is not in known_columns.
    Returns None if no unexpected columns exist.
    Equivalent to Databricks rescuedDataColumn per-record behavior.

    Parameters
    ----------
    record        : single JSON record as a dict
    known_columns : set of expected column names (BRONZE_COLUMNS)

    Returns
    -------
    str or None
        JSON string of unexpected key-value pairs, or None if no drift.
    """
    unexpected = {k: v for k, v in record.items() if k not in known_columns}
    return json.dumps(unexpected, ensure_ascii=False) if unexpected else None


def _log_drift_event(year_month: str, rescued_count: int, sample: str) -> None:
    """
    Append a schema drift event to schema_drift_log.json.
    Local equivalent of INSERT INTO ops_quality_events in Databricks.

    Parameters
    ----------
    year_month    : affected month (e.g. "2026-04")
    rescued_count : number of records with unexpected columns
    sample        : sample rescued_data blob (truncated to 300 chars)
    """
    event = {
        "batch_id"      : BATCH_ID,
        "source_id"     : SOURCE_ID,
        "object"        : "bronze_compras_contratos",
        "event_type"    : "SCHEMA_DRIFT",
        "severity"      : "WARN",
        "action"        : "CONTINUE",
        "year_month"    : year_month,
        "rescued_count" : rescued_count,
        "sample"        : sample[:300],
        "logged_at"     : datetime.now(timezone.utc).isoformat(),
    }
    existing = []
    if DRIFT_LOG_PATH.exists():
        with open(DRIFT_LOG_PATH, "r", encoding="utf-8") as f:
            try:
                existing = json.load(f)
            except json.JSONDecodeError:
                existing = []
    existing.append(event)
    DRIFT_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(DRIFT_LOG_PATH, "w", encoding="utf-8") as f:
        json.dump(existing, f, ensure_ascii=False, indent=2)


def process_month(year_month: str, con) -> dict:
    """
    Read the raw JSON for a given month, apply Bronze transformations,
    build _rescued_data for schema drift capture, and write Parquet.

    Parameters
    ----------
    year_month : month string e.g. "2026-04"
    con        : DuckDB connection

    Returns
    -------
    dict
        Processing result with keys: year_month, status, records,
        rescued_count, file, error.

    Notes
    -----
    Transformations applied (Bronze layer only):
    - All source columns cast to VARCHAR
    - unidadesRequisitantes: serialized via to_json() if STRUCT/LIST
    - _rescued_data: JSON blob of unexpected columns per record
    - Technical columns: _source_id, _batch_id, _ingested_at,
                         _source_file, _year_month
    """
    # Flat file structure — matches 04_bootstrap_compras.py output
    source_file = RAW_ROOT / f"{year_month}.json"

    if not source_file.exists():
        return {
            "year_month"   : year_month,
            "status"       : "MISSING",
            "records"      : 0,
            "rescued_count": 0,
            "file"         : None,
            "error"        : f"File not found: {source_file}",
        }

    partition_dir = BRONZE_ROOT / f"_year_month={year_month}"
    partition_dir.mkdir(parents=True, exist_ok=True)
    output_file  = partition_dir / "data.parquet"
    source_path  = to_sql_path(source_file)
    output_path  = to_sql_path(output_file)

    try:
        known_cols  = set(BRONZE_COLUMNS)
        json_cols   = _inferred_as_json(source_path, con)
        source_cols = _get_source_columns(source_path, con)
        unexpected  = source_cols - known_cols

        # ---------------------------------------------------------------
        # Fast path — no schema drift detected
        # Pure DuckDB SQL — no Python per-record processing
        # ---------------------------------------------------------------
        if not unexpected:
            col_expressions = []
            for col in BRONZE_COLUMNS:
                if col in json_cols:
                    col_expressions.append(
                        f'CAST(to_json("{col}") AS VARCHAR) AS "{col}"'
                    )
                else:
                    col_expressions.append(
                        f'CAST("{col}" AS VARCHAR) AS "{col}"'
                    )
            cols_sql = ",\n                        ".join(col_expressions)

            con.execute(f"""
                COPY (
                    SELECT
                        {cols_sql},
                        NULL::VARCHAR              AS _rescued_data,
                        '{SOURCE_ID}'            AS _source_id,
                        '{BATCH_ID}'             AS _batch_id,
                        CURRENT_TIMESTAMP          AS _ingested_at,
                        '{source_path}'          AS _source_file,
                        '{year_month}'           AS _year_month
                    FROM read_json_auto('{source_path}',
                        maximum_object_size = 33554432
                    )
                ) TO '{output_path}'
                (FORMAT PARQUET, COMPRESSION SNAPPY)
            """)

            records = con.execute(
                f"SELECT COUNT(*) FROM read_parquet('{output_path}')"
            ).fetchone()[0]

            return {
                "year_month"   : year_month,
                "status"       : "SUCCESS",
                "records"      : records,
                "rescued_count": 0,
                "file"         : str(output_file),
                "error"        : None,
            }

        # ---------------------------------------------------------------
        # Slow path — schema drift detected
        # Python per-record processing to build _rescued_data
        # This path is only taken when drift occurs
        # ---------------------------------------------------------------
        with open(source_file, "r", encoding="utf-8") as f:
            raw_data = json.load(f)

        enriched_records = []
        rescued_count    = 0
        sample_rescued   = None

        for record in raw_data:
            rescued = _build_rescued_data(record, known_cols)
            if rescued:
                rescued_count += 1
                if sample_rescued is None:
                    sample_rescued = rescued

            row = {}
            for col in BRONZE_COLUMNS:
                val = record.get(col)
                row[col] = json.dumps(val, ensure_ascii=False) \
                    if isinstance(val, (dict, list)) \
                    else (None if val is None else str(val))

            row["_rescued_data"] = rescued
            row["_source_id"]    = SOURCE_ID
            row["_batch_id"]     = BATCH_ID
            row["_ingested_at"]  = datetime.now(timezone.utc).isoformat()
            row["_source_file"]  = source_path
            row["_year_month"]   = year_month
            enriched_records.append(row)

        con.execute("CREATE OR REPLACE TEMP TABLE _drift_staging AS "
                    "SELECT * FROM enriched_records")
        con.execute(f"""
            COPY (SELECT * FROM _drift_staging)
            TO '{output_path}'
            (FORMAT PARQUET, COMPRESSION SNAPPY)
        """)

        records = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{output_path}')"
        ).fetchone()[0]

        if rescued_count > 0:
            _log_drift_event(year_month, rescued_count, sample_rescued or "")

        return {
            "year_month"   : year_month,
            "status"       : "SUCCESS",
            "records"      : records,
            "rescued_count": rescued_count,
            "file"         : str(output_file),
            "error"        : None,
        }

    except Exception as e:
        return {
            "year_month"   : year_month,
            "status"       : "ERROR",
            "records"      : 0,
            "rescued_count": 0,
            "file"         : None,
            "error"        : str(e),
        }


# ---------------------------------------------------------------------------
# Execute — process all available months
# ---------------------------------------------------------------------------
available_months = sorted([
    f.stem for f in RAW_ROOT.glob("*.json")
])

if not available_months:
    raise FileNotFoundError(
        f"No JSON files found in {RAW_ROOT}\n"
        "Run 04_bootstrap_compras.py first."
    )

print(f"Months available : {len(available_months)}")
print(f"Range            : {available_months[0]} -> {available_months[-1]}")
print()

BRONZE_ROOT.mkdir(parents=True, exist_ok=True)
results = []
con = get_connection()  # in-memory — Parquet I/O does not require persistent DB

for i, ym in enumerate(available_months, 1):
    res = process_month(ym, con)
    results.append(res)
    icon         = "OK " if res["status"] == "SUCCESS" else "ERR"
    drift_suffix = f" — {res['rescued_count']:,} rescued" if res.get("rescued_count", 0) > 0 else ""
    print(f"  [{icon}] [{i:02d}/{len(available_months)}] {ym} "
          f"— {res['status']} — {res['records']:>6,} records{drift_suffix}")

con.close()

total_records = sum(r["records"] for r in results)
success_count = sum(1 for r in results if r["status"] == "SUCCESS")
error_count   = sum(1 for r in results if r["status"] == "ERROR")
missing_count = sum(1 for r in results if r["status"] == "MISSING")
drift_months  = [r["year_month"] for r in results if r.get("rescued_count", 0) > 0]
total_rescued = sum(r.get("rescued_count", 0) for r in results)

print()
print(f"  Success  : {success_count}")
print(f"  Error    : {error_count}")
print(f"  Missing  : {missing_count}")
print(f"  Records  : {total_records:,}")
print(f"  Drift    : {len(drift_months)} month(s) — {total_rescued:,} rescued records")
if drift_months:
    print(f"  Months   : {drift_months}")
if error_count:
    print()
    print("  Errors:")
    for r in results:
        if r["status"] == "ERROR":
            print(f"    {r['year_month']} — {r['error']}")


## Step 4 — Validation

In [ ]:
suite = CheckSuite("bronze_compras_contratos")
con_val = get_connection()  # fresh connection for validation reads
bronze_glob = to_sql_path(BRONZE_ROOT / "**" / "*.parquet")

# Check 1 — no processing errors or missing files
suite.add(
    "No processing errors or missing files",
    error_count == 0 and missing_count == 0,
    f"errors={error_count}, missing={missing_count}",
)

# Check 2 — all partition directories exist
all_partitions_exist = all(
    (BRONZE_ROOT / f"_year_month={r['year_month']}" / "data.parquet").exists()
    for r in results if r["status"] == "SUCCESS"
)
suite.add("All partition Parquet files exist", all_partitions_exist, "")

# Check 3 — no empty partitions
suite.add(
    "No empty partitions",
    all(r["records"] > 0 for r in results if r["status"] == "SUCCESS"),
    "",
)

# Check 4 — total record count matches source
bronze_total = con_val.execute(
    f"SELECT COUNT(*) FROM read_parquet('{bronze_glob}', hive_partitioning=true)"
).fetchone()[0]
suite.add(
    "Record count matches source",
    total_records == bronze_total,
    f"source={total_records:,}  bronze={bronze_total:,}",
)

# Check 5 — correct number of columns
actual_cols = con_val.execute(f"""
    SELECT COUNT(*) FROM (
        DESCRIBE SELECT * FROM read_parquet('{bronze_glob}', hive_partitioning=true)
        LIMIT 0
    )
""").fetchone()[0]
suite.add(
    "Column count correct",
    actual_cols == len(ALL_COLUMNS),
    f"found={actual_cols}  expected={len(ALL_COLUMNS)}",
)

# Check 6 — all technical columns present
existing_cols = {
    row[0] for row in con_val.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{bronze_glob}', hive_partitioning=true) LIMIT 0"
    ).fetchall()
}
missing_tech = [c for c in TECHNICAL_COLUMNS if c not in existing_cols]
suite.add(
    "All technical columns present",
    not missing_tech,
    f"missing: {missing_tech}" if missing_tech else "all present",
)

# Check 7 — schema drift monitored (WARN, never FAIL — pipeline always continues)
suite.add(
    "Schema drift monitored",
    True,  # always passes — drift is a warning, not a blocker
    f"{len(drift_months)} month(s) with rescued data"
    + (f": {drift_months}" if drift_months else " — none detected"),
)

# Check 8 — niFornecedor not null in any partition
null_ni = con_val.execute(
    f"SELECT COUNT(*) FROM read_parquet('{bronze_glob}', hive_partitioning=true)"
    " WHERE niFornecedor IS NULL"
).fetchone()[0]
suite.add(
    "niFornecedor null rate acceptable",
    null_ni / bronze_total < 0.05 if bronze_total else True,
    f"{null_ni:,} nulls ({null_ni/bronze_total*100:.2f}%)" if bronze_total else "n/a",
)

con_val.close()
suite.report()
suite.assert_all_pass()


## Step 5 — Ops registration

In [ ]:
files_written = sum(1 for r in results if r["status"] == "SUCCESS")
bytes_written = sum(
    (BRONZE_ROOT / f"_year_month={r['year_month']}" / "data.parquet").stat().st_size
    for r in results if r["status"] == "SUCCESS"
)
period        = f"{available_months[0]}/{available_months[-1]}"
batch_status  = "SUCCESS" if error_count == 0 else "PARTIAL"
errors_detail = (
    "; ".join(f"{r['year_month']}: {r['error'][:60]}" for r in results if r["status"] == "ERROR")
    if error_count > 0 else None
)

entry = make_entry(
    source_id    = SOURCE_ID,
    period       = period,
    status       = batch_status,
    records      = total_records,
    bytes_written = bytes_written,
    started_at   = STARTED_AT,
    finished_at  = datetime.now(timezone.utc).isoformat(),
    error_message = errors_detail,
    # Bronze-specific extra fields (Opção A — **extra_fields in make_entry)
    batch_id         = BATCH_ID,
    layer            = "bronze",
    object           = "compras_contratos",
    files            = files_written,
    has_rescued_data = len(drift_months) > 0,
    drift_months     = len(drift_months),
)

append_log(LOG_PATH, entry)

print(f"Batch registered   : {BATCH_ID}")
print(f"Status             : {batch_status}")
print(f"Records            : {total_records:,}")
print(f"Files              : {files_written}")
print(f"Bytes written      : {bytes_written / (1024*1024):.1f} MB")
print(f"Period             : {period}")
print(f"Has rescued data   : {len(drift_months) > 0}")
print(f"Log                : {LOG_PATH}")
